# Step 1: Load Cleaned Dataset

Load the cleaned dataset generated from the EDA notebook.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/cleaned_paysim_data.csv')
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_diff_orig,balance_diff_dest
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,9839.64,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1864.28,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,181.00,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,181.00,-21182.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,11668.14,0.0


# Step 2: Basic Validation

Check shape and null values to ensure dataset integrity.

In [ ]:
df.shape
df.isnull().sum()

step                 0
type                 0
amount               0
nameOrig             0
oldbalanceOrg        0
newbalanceOrig       0
nameDest             0
oldbalanceDest       0
newbalanceDest       0
isFraud              0
isFlaggedFraud       0
balance_diff_orig    0
balance_diff_dest    0
dtype: int64

# Step 3: Drop Irrelevant Columns

Remove ID-like columns that do not contribute to modeling.

In [ ]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_diff_orig,balance_diff_dest
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,9839.64,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1864.28,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,181.00,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,181.00,-21182.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,11668.14,0.0


In [ ]:
df.drop(['nameOrig', 'nameDest'], axis=1, inplace=True)

In [ ]:
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['newbalanceDest'] - df['oldbalanceDest']

# Step 5: Encoding

Convert categorical variables into numerical format using one-hot encoding.

In [ ]:
df = pd.get_dummies(df, columns=['type'], drop_first=True)
df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_diff_orig,balance_diff_dest,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,0,181.00,0.0,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,181.00,-21182.0,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,False,False,True,False


# Step 6: Split Features and Target

Separate input features and target variable.

In [ ]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Step 7: Train-Test Split

Split dataset into training and temporary sets.

# Step 8: Validation Split

Split temporary data into validation and test sets.

# Step 9: Handle Class Imbalance

Instead of aggressive undersampling, we reduce imbalance while preserving useful data.

We use RandomUnderSampler with controlled sampling strategy to avoid excessive data loss.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

print("Original:", X.shape)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Temp:", X_temp.shape)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Val:", X_val.shape, "Test:", X_test.shape)

from imblearn.combine import SMOTEENN

sm = SMOTEENN(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

Original: (6362620, 13)
Train: (4453834, 13) Temp: (1908786, 13)
Val: (954393, 13) Test: (954393, 13)


In [ ]:
X_train.shape, X_val.shape, X_test.shape


((8869960, 13), (954393, 13), (954393, 13))

# Step 11: Save Processed Dataset

Save processed datasets for model training.

In [ ]:
# Training Data
X_train.to_csv('../data/X_train.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)

# Validation Data
X_val.to_csv('../data/X_val.csv', index=False)
y_val.to_csv('../data/y_val.csv', index=False)

# Test Data
X_test.to_csv('../data/X_test.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)